In [ ]:
import pyspark.sql.types as t
from gentropy.common.session import Session
from pyspark.sql import functions as f


Loading BokehJS ...

/Users/dc16/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [2]:
session = Session(extended_spark_conf={"spark.driver.memory": "10g"})


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/26 14:45:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
path_to_release_folder = "../../data/25.06/"

output_path = "../../data/intermediate_files/"


# Therapeutic area hierarchy

If a disease/trait has more than one therapeutic area at the top of its ontology tree, the first one from this list is taken, otherwise it is assigned 'other'.


In [4]:
therapy_area_hierarchy = {
    "EFO_0001444": "measurement",
    "MONDO_0045024": "cancer or benign tumor",
    "EFO_0005741": "infectious disease",
    "OTAR_0000009": "injury, poisoning or other complication",
    "OTAR_0000014": "pregnancy or perinatal disease",
    "MONDO_0024458": "disorder of visual system",
    "EFO_0000319": "cardiovascular disease",
    "EFO_0009605": "pancreas disease",
    "EFO_0000540": "immune system disease",
    "EFO_0010282": "gastrointestinal disease",
    "OTAR_0000017": "reproductive system or breast disease",
    "EFO_0010285": "integumentary system disease",
    "EFO_0001379": "endocrine system disease",
    "OTAR_0000010": "respiratory or thoracic disease",
    "EFO_0009690": "urinary system disease",
    "OTAR_0000006": "musculoskeletal or connective tissue disease",
    "MONDO_0021205": "disorder of ear",
    "EFO_0005803": "hematologic disease",
    "EFO_0000618": "nervous system disease",
    "MONDO_0002025": "psychiatric disorder",
    "OTAR_0000020": "nutritional or metabolic disease",
    "OTAR_0000018": "genetic, familial or congenital disease",
    "EFO_0003765": "sign or symptom",  # Not a therapeutic area - is descendant of phenotype
}


In [5]:
studies = session.spark.read.parquet(f"{path_to_release_folder}/output/study")
cs_lead_variant_effect = session.spark.read.parquet(f"{output_path}/lead_variant_effect")


# Study-Index with therapeutic areas


In [6]:
# This udf extracts the FIRST therapeutic area, as per hierarchy list, for each diseaseId
@f.udf(t.StringType())
def get_first_matching_therapeutic_area(therapeutic_areas_list):
    if therapeutic_areas_list is None:
        return None
    for ta in therapy_area_hierarchy:
        if ta in therapeutic_areas_list:
            return ta
    return None


# These lines create a dictionary of diseaseId to primary therapeutic area
efo_ta = (
    session.spark.read.parquet(f"{path_to_release_folder}output/disease/disease.parquet")
    .select(
        "id",
        "ancestors",
    )
    .withColumn(
        "primaryTherapeuticArea",
        get_first_matching_therapeutic_area(f.col("ancestors")),
    )
    .withColumn(
        "primaryTherapeuticArea",
        f.when(f.col("primaryTherapeuticArea").isNull(), f.lit("other")).otherwise(f.col("primaryTherapeuticArea")),
    )
    .join(
        studies.select(f.explode("diseaseIds").alias("efo")),
        f.col("id") == f.col("efo"),
        "semi",
    )
)
efo_ta_lookup = efo_ta.select("id", "primaryTherapeuticArea").collect()
efo_ta_dict = {row["id"]: row["primaryTherapeuticArea"] for row in efo_ta_lookup}


# This udf takes a diseaseIds arrays and creates an array of mapped therapeutic areas
@f.udf(t.ArrayType(t.StringType()))
def map_efos_to_therapeutic_areas(efo_ids):
    if efo_ids is None:
        return None
    lookup_dict = efo_ta_dict
    mapped_areas = []
    for efo_id in efo_ids:
        mapped_areas.append(lookup_dict.get(efo_id, None))
        mapped_areas = list(set(area for area in mapped_areas if area is not None))
    return mapped_areas


In [7]:
gwas = (
    studies.filter(f.col("studyType") == "gwas")
    .withColumn("mappedTherapeuticAreas", map_efos_to_therapeutic_areas(f.col("diseaseIds")))
    .withColumn("measurement", f.array_contains("mappedTherapeuticAreas", "EFO_0001444"))
    .withColumn(
        "binaryLessCases",
        f.when(f.col("nCases") < f.col("nControls"), True).otherwise(False),
    )
    .withColumns(
        {
            "cancerOrBenignTumor": f.when(f.array_contains("mappedTherapeuticAreas", "MONDO_0045024"), 1).otherwise(0),
            "infectiousDisease": f.when(f.array_contains("mappedTherapeuticAreas", "EFO_0005741"), 1).otherwise(0),
            "pregnancyOrPerinatalDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000014"), 1
            ).otherwise(0),
            "disorderOfVisualSystem": f.when(f.array_contains("mappedTherapeuticAreas", "MONDO_0024458"), 1).otherwise(
                0
            ),
            "cardiovascularDisease": f.when(f.array_contains("mappedTherapeuticAreas", "EFO_0000319"), 1).otherwise(0),
            "pancreasDisease": f.when(f.array_contains("mappedTherapeuticAreas", "EFO_0009605"), 1).otherwise(0),
            "gastrointestinalDisease": f.when(f.array_contains("mappedTherapeuticAreas", "EFO_0010282"), 1).otherwise(
                0
            ),
            "reproductiveSystemOrBreastDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000017"), 1
            ).otherwise(0),
            "integumentarySystemDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0010285"), 1
            ).otherwise(0),
            "endocrineSystemDisease": f.when(f.array_contains("mappedTherapeuticAreas", "EFO_0001379"), 1).otherwise(0),
            "respiratoryOrThoracicDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000010"), 1
            ).otherwise(0),
            "urinarySystemDisease": f.when(f.array_contains("mappedTherapeuticAreas", "EFO_0009690"), 1).otherwise(0),
            "musculoskeletalOrConnectiveTissueDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000006"), 1
            ).otherwise(0),
            "disorderOfEar": f.when(f.array_contains("mappedTherapeuticAreas", "MONDO_0021205"), 1).otherwise(0),
            "immuneSystemDisease": f.when(f.array_contains("mappedTherapeuticAreas", "EFO_0000540"), 1).otherwise(0),
            "hematologicDisease": f.when(f.array_contains("mappedTherapeuticAreas", "EFO_0005803"), 1).otherwise(0),
            "nervousSystemDisease": f.when(f.array_contains("mappedTherapeuticAreas", "EFO_0000618"), 1).otherwise(0),
            "psychiatricDisorder": f.when(f.array_contains("mappedTherapeuticAreas", "MONDO_0002025"), 1).otherwise(0),
            "nutritionalOrMetabolicDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000020"), 1
            ).otherwise(0),
            "geneticFamilialOrCongenitalDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000018"), 1
            ).otherwise(0),
            "injuryPoisoningOrOtherComplication": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000009"), 1
            ).otherwise(0),
            "signOrSymptom": f.when(f.array_contains("mappedTherapeuticAreas", "EFO_0003765"), 1).otherwise(0),
            "other": f.when(f.array_contains("mappedTherapeuticAreas", "other"), 1).otherwise(0),
        }
    )
    .withColumn(
        "totalTherapeuticAreas",
        f.col("cancerOrBenignTumor")
        + f.col("infectiousDisease")
        + f.col("pregnancyOrPerinatalDisease")
        + f.col("disorderOfVisualSystem")
        + f.col("cardiovascularDisease")
        + f.col("pancreasDisease")
        + f.col("gastrointestinalDisease")
        + f.col("reproductiveSystemOrBreastDisease")
        + f.col("integumentarySystemDisease")
        + f.col("endocrineSystemDisease")
        + f.col("respiratoryOrThoracicDisease")
        + f.col("urinarySystemDisease")
        + f.col("musculoskeletalOrConnectiveTissueDisease")
        + f.col("disorderOfEar")
        + f.col("immuneSystemDisease")
        + f.col("hematologicDisease")
        + f.col("nervousSystemDisease")
        + f.col("psychiatricDisorder")
        + f.col("nutritionalOrMetabolicDisease")
        + f.col("geneticFamilialOrCongenitalDisease")
        + f.col("injuryPoisoningOrOtherComplication")
        + f.col("signOrSymptom")
        + f.col("other"),
    )
)


In [9]:
gwas.write.parquet(
    f"{output_path}/gwas_w_therapeutic_areas",
    mode="overwrite",
)


25/11/26 14:00:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


# Qualifying studies


In [ ]:
si_ta = session.spark.read.parquet(f"{output_path}/gwas_w_therapeutic_areas").cache()


In [ ]:
qualifying_studies = (
    si_ta.filter(f.col("binaryLessCases"))
    .filter(~f.col("measurement"))
    .filter(f.col("nSamples") >= 1000)
    .filter((f.col("nCases") / f.col("nSamples")) >= 0.001)
    .cache()
)
qualifying_studies.count()


15768

Removing a problematic publication:


In [ ]:
qualifying_studies = qualifying_studies.filter(
    (~f.col("pubmedId").isin(["40069456"])) | (f.col("pubmedId").isNull())
).cache()
qualifying_studies.count()


15730

In [ ]:
qualifying_studies.write.parquet(
    f"{output_path}/qualifying_gwas_studies",
    mode="overwrite",
)


# Qualifying measurement studies


In [ ]:
# Removing protein measurements and microbiome descendants
# EFO_0007882 - microbiome
# EFO_0004747 - protein measurement

proteins_and_microbiome = (
    session.spark.read.parquet(f"{path_to_release_folder}/output/disease/disease.parquet")
    .select("id", "descendants")
    .filter(f.col("id").isin(["EFO_0007882", "EFO_0004747"]))
    .select(f.explode("descendants"))
)
proteins_and_microbiome_ids = [row["col"] for row in proteins_and_microbiome.collect()]
proteins_and_microbiome_ids.extend(["EFO_0007882", "EFO_0004747"])


In [ ]:
qualifying_measurements = (
    si_ta.filter(f.col("measurement"))
    .filter(~f.col("binaryLessCases"))
    .filter(f.size(f.array_intersect(f.col("diseaseIds"), f.lit(proteins_and_microbiome_ids))) == 0)
)


In [ ]:
qualifying_measurements.count()


61885

In [ ]:
qualifying_measurements.write.parquet(
    f"{output_path}/qualifying_measurement_studies",
    mode="overwrite",
)


# Qualified credible sets generation


In [17]:
qualifying_studies = session.spark.read.parquet(f"{output_path}/qualifying_gwas_studies")
l2g_feature_matrix = session.spark.read.parquet(f"{output_path}/l2g_feature_matrix")
replicated_cs = session.spark.read.parquet(f"{output_path}/list_of_gwas_replicated_CSs.parquet")


In [18]:
features = (
    l2g_feature_matrix.groupBy("studyLocusId")
    .agg(
        f.max("eQtlColocH4Maximum").alias("eqtlH4"),
        f.max("eQtlColocClppMaximum").alias("eqtlCLPP"),
        f.max("pQtlColocH4Maximum").alias("pqtlH4"),
        f.max("pQtlColocClppMaximum").alias("pqtlCLPP"),
        f.max("sQtlColocH4Maximum").alias("sqtlH4"),
        f.max("sQtlColocClppMaximum").alias("sqtlCLPP"),
        f.max("vepMaximum").alias("vep"),
    )
    .join(
        replicated_cs.withColumn("replicated", f.lit(1)),
        "studyLocusId",
        "outer",
    )
    .withColumn(
        "replicated",
        f.when(f.col("replicated").isNull(), 0).otherwise(f.col("replicated")),
    )
    .persist()
)
features.show()


+--------------------+----------+------------+---------+-----------+----------+-----------+----+----------+
|        studyLocusId|    eqtlH4|    eqtlCLPP|   pqtlH4|   pqtlCLPP|    sqtlH4|   sqtlCLPP| vep|replicated|
+--------------------+----------+------------+---------+-----------+----------+-----------+----+----------+
|0005218bc3a62e387...|       0.0|         0.0|      0.0|        0.0|       0.0|        0.0|0.66|         0|
|002462a2da2f7c279...|       0.0|         0.0|      0.0|        0.0|       0.0|        0.0| 0.0|         0|
|00274cac95947bd00...|       0.0| 0.019822257|      0.0|        0.0|       0.0|        0.0| 0.1|         0|
|005bc8624f8dd7f7c...|       1.0|        0.95|      0.0|        0.0|       1.0|       0.95|0.33|         1|
|0064268fb58ddabbb...|       0.0|         0.0|      0.0|        0.0|       0.0|        0.0| 0.0|         0|
|008b97f651f444888...|0.97240156| 0.032121874|      0.0|        0.0|       0.0|        0.0| 0.1|         0|
|009c65f69b7705d4c...|0.9760

In [19]:
(
    cs_lead_variant_effect.join(
        qualifying_studies.select("studyId", "nCases", "nControls", "nSamples"),
        "studyId",
        "inner",
    ).count()
)


96581

In [20]:
first_filter_cs = (
    cs_lead_variant_effect.join(
        qualifying_studies.select("studyId", "nCases", "nControls", "nSamples"),
        "studyId",
        "inner",
    )
    .filter(2 * f.col("majorLdPopulationMaf.value") * f.col("nSamples") >= 20)
    .filter(f.abs("rescaledStatistics.estimatedBeta") <= 3)
    .persist()
)
first_filter_cs.count()


77455

In [21]:
common_cs = first_filter_cs.filter(f.col("majorLdPopulationMaf.value") > 0.01).persist()
common_cs.count()


69090

In [ ]:
rare_cs = (
    first_filter_cs.filter(f.col("majorLdPopulationMaf.value") <= 0.01)
    .join(
        features.filter(
            (f.col("eqtlH4") >= 0.8)
            | (f.col("eqtlCLPP") >= 0.01)
            | (f.col("pqtlH4") >= 0.8)
            | (f.col("pqtlCLPP") >= 0.01)
            | (f.col("sqtlH4") >= 0.8)
            | (f.col("sqtlCLPP") >= 0.01)
            | (f.col("vep") >= 0.66)
            | (f.col("replicated") == 1)
        ),
        "studyLocusId",
        "semi",
    )
    .persist()
)
rare_cs.count()


1935

In [ ]:
qualifying_credible_sets = common_cs.unionByName(rare_cs)
qualifying_credible_sets.count()


71025

In [ ]:
qualifying_credible_sets.write.parquet(
    f"{output_path}/qualifying_credible_sets",
    mode="overwrite",
)


# Qualifying Measurement Credible Sets


In [ ]:
qualifying_measurements = session.spark.read.parquet(f"{output_path}/qualifying_measurement_studies")


In [ ]:
first_filter_measurements = (
    cs_lead_variant_effect.join(qualifying_measurements.select("studyId", "nSamples"), "studyId", "inner")
    .filter(2 * f.col("majorLdPopulationMaf.value") * f.col("nSamples") >= 20)
    .filter(f.abs("rescaledStatistics.estimatedBeta") <= 3)
    .persist()
)
first_filter_measurements.count()


473831

In [ ]:
common_measurements = first_filter_measurements.filter(f.col("majorLdPopulationMaf.value") > 0.01).persist()
common_measurements.count()


436706

In [33]:
rare_measurements = (
    first_filter_measurements.filter(f.col("majorLdPopulationMaf.value") <= 0.01)
    .join(
        features.filter(
            (f.col("eqtlH4") >= 0.8)
            | (f.col("eqtlCLPP") >= 0.01)
            | (f.col("pqtlH4") >= 0.8)
            | (f.col("pqtlCLPP") >= 0.01)
            | (f.col("sqtlH4") >= 0.8)
            | (f.col("sqtlCLPP") >= 0.01)
            | (f.col("vep") >= 0.66)
            | (f.col("replicated") == 1)
        ),
        "studyLocusId",
        "semi",
    )
    .persist()
)
rare_measurements.count()


25/11/26 15:17:35 WARN CacheManager: Asked to cache already cached data.


13651

In [ ]:
qualifying_measurement_credible_sets = common_measurements.unionByName(rare_measurements)
qualifying_measurement_credible_sets.count()


450357

In [ ]:
qualifying_measurement_credible_sets.write.parquet(
    f"{output_path}/qualifying_measurement_credible_sets",
    mode="overwrite",
)
